In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Download and Load the Dataset

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/Ardeshir-Shon/dataMiningLabTutorial/main/kzThiner.csv')

# Data Exploration

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

# Preprocessing

In [ ]:
df.isna().sum()

In [ ]:
df['event_time'] = pd.to_datetime(df['event_time'])

In [ ]:
missing_attributes = ['category_id', 'category_code', 'brand', 'price', 'user_id']

missing_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

column_transformer = ColumnTransformer([
    ('missing', missing_pipeline, missing_attributes),
], remainder='passthrough')

In [ ]:
transformed_data = column_transformer.fit_transform(df)
transformed_df = pd.DataFrame(transformed_data, columns=column_transformer.get_feature_names_out())

transformed_df.head()

# Feature Engineering for RFM

In [ ]:
# Calculate the number of days between a customers first and last order
transformed_df['recency'] = (transformed_df.groupby('missing__user_id')['remainder__event_time'].transform('max') - transformed_df.groupby('missing__user_id')['remainder__event_time'].transform('min')).dt.days

# Calculate the number of purchases a customer has made
transformed_df['frequency'] = transformed_df.groupby('missing__user_id')['remainder__event_time'].transform('count')

# Calculate the total amount of money spent by a customer
transformed_df['monetary'] = transformed_df.groupby('missing__user_id')['missing__price'].transform('sum')

transformed_df.head()

# Implement K-means

In [ ]:
# Apply K-Means clustering to the RFM data
X = transformed_df[['recency', 'frequency', 'monetary']].values
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X)

# Elbow Method

In [ ]:
# Apply the elbow method to find the optimal number of clusters
inertia = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertia.append(kmeans.inertia_)

plt.plot(range(1, 11), inertia)
plt.title('Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.show()